# Mini_Assignment_2_Shradha_Bavalatti

In [1]:
# Install PySpark
from google.colab import files
uploaded = files.upload()
import pandas as pd
df_pd = pd.read_csv("Flight Dataset.csv")
df_pd.head()

Saving Flight Dataset.csv to Flight Dataset.csv


,FL_DATE,DEP_DELAY,ARR_DELAY,AIR_TIME,DISTANCE,DEP_TIME,ARR_TIME
0,1/1/2006,5,19,350,2475,9.083333,12.483334
1,1/2/2006,167,216,343,2475,11.783334,15.766666
2,1/3/2006,-7,-2,344,2475,8.883333,12.133333
3,1/4/2006,-5,-13,331,2475,8.916667,11.950000
4,1/5/2006,-3,-17,321,2475,8.950000,11.883333


In [2]:
!pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
spark = SparkSession.builder.getOrCreate()
df = spark.createDataFrame(df_pd)
df.show(5)
df.printSchema()

+--------+---------+---------+--------+--------+---------+---------+
| FL_DATE|DEP_DELAY|ARR_DELAY|AIR_TIME|DISTANCE| DEP_TIME| ARR_TIME|
+--------+---------+---------+--------+--------+---------+---------+
|1/1/2006|        5|       19|     350|    2475| 9.083333|12.483334|
|1/2/2006|      167|      216|     343|    2475|11.783334|15.766666|
|1/3/2006|       -7|       -2|     344|    2475| 8.883333|12.133333|
|1/4/2006|       -5|      -13|     331|    2475| 8.916667|    11.95|
|1/5/2006|       -3|      -17|     321|    2475|     8.95|11.883333|
+--------+---------+---------+--------+--------+---------+---------+
only showing top 5 rows

root
 |-- FL_DATE: string (nullable = true)
 |-- DEP_DELAY: long (nullable = true)
 |-- ARR_DELAY: long (nullable = true)
 |-- AIR_TIME: long (nullable = true)
 |-- DISTANCE: long (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)



## Task 1 Create a function that gives back how many flights arrived earlier than expected.

In [4]:
from pyspark.sql.functions import col

def count_early_arrivals(dataframe):
    df_early = dataframe.filter(col("ARR_DELAY") < 0)
    early_count = df_early.count()
    print(f"Number of flights that arrived earlier than expected: {early_count}")
    return df_early

# Run Task 1
df_early = count_early_arrivals(df)

# Display sample output
df_early.show(10)
df_early.printSchema()

Number of flights that arrived earlier than expected: 534655
+---------+---------+---------+--------+--------+--------+---------+
|  FL_DATE|DEP_DELAY|ARR_DELAY|AIR_TIME|DISTANCE|DEP_TIME| ARR_TIME|
+---------+---------+---------+--------+--------+--------+---------+
| 1/3/2006|       -7|       -2|     344|    2475|8.883333|12.133333|
| 1/4/2006|       -5|      -13|     331|    2475|8.916667|    11.95|
| 1/5/2006|       -3|      -17|     321|    2475|    8.95|11.883333|
| 1/6/2006|       -4|      -32|     320|    2475|8.933333|11.633333|
| 1/8/2006|       -3|       -2|     346|    2475|    8.95|12.133333|
|1/10/2006|       -7|      -21|     334|    2475|8.883333|11.816667|
|1/11/2006|        8|      -10|     321|    2475|9.133333|     12.0|
|1/12/2006|       -5|      -27|     321|    2475|8.916667|11.716666|
|1/13/2006|       -7|       -6|     327|    2475|8.883333|12.066667|
|1/16/2006|       -4|      -14|     329|    2475|8.933333|11.933333|
+---------+---------+---------+--------+--

## Task 2 – Create a function that determines the typical departure time for flights over 2000 miles.

In [5]:
from pyspark.sql.functions import col, avg

def typical_departure_time_long_flights(dataframe):

    # Filter flights with distance > 2000 miles
    long_flights = dataframe.filter(col("DISTANCE") > 2000)

    # Calculate the average departure time
    avg_dep_time = (
        long_flights.agg(avg(col("DEP_TIME")).alias("typical_dep_time"))
        .collect()[0]["typical_dep_time"]
    )

    print("Typical Departure Time for Long Flights (>2000 miles)")
    print(f"Typical (average) departure time: {avg_dep_time:.2f} hours")
    return avg_dep_time

typical_time = typical_departure_time_long_flights(df)

Typical Departure Time for Long Flights (>2000 miles)
Typical (average) departure time: 13.97 hours


## Task 3 – Create a function that gives back the proportion of flights that have arrival delays longer than 60 minutes.

In [6]:
from pyspark.sql.functions import col, count

def proportion_long_arrival_delays(dataframe):

    # Total number of flights
    total_flights = dataframe.count()

    # Flights with ARR_DELAY > 60
    delayed_flights = dataframe.filter(col("ARR_DELAY") > 60).count()

    # Calculate the proportion
    proportion = delayed_flights / total_flights if total_flights > 0 else 0

    print(f"Total Flights: {total_flights}")
    print(f"Flights Delayed > 60 Minutes: {delayed_flights}")
    print(f"Proportion: {proportion:.4f} ({proportion * 100:.2f}%)")

    return proportion

proportion_delayed = proportion_long_arrival_delays(df)

Total Flights: 1000000
Flights Delayed > 60 Minutes: 53066
Proportion: 0.0531 (5.31%)


## Task 4 –  Create a function that gives the average airtime for flights that left earlier than 9:00 am.

In [7]:
from pyspark.sql.functions import col, avg

def average_airtime_early_departures(dataframe):
    # Filter flights that departed before 9:00 AM
    early_flights = dataframe.filter(col("DEP_TIME") < 900)

    # Calculate the average airtime
    avg_airtime = (
        early_flights.agg(avg(col("AIR_TIME")).alias("avg_airtime"))
        .collect()[0]["avg_airtime"]
    )

    print(f"Average Airtime: {avg_airtime:.2f} minutes")

    return avg_airtime

average_early_airtime = average_airtime_early_departures(df)

Average Airtime: 105.81 minutes


## Task 5 – Create a function that determines the maximum arrival delay for flights that did not experience a delay upon departure.

In [8]:
from pyspark.sql.functions import col, max as spark_max

def max_arrival_delay_no_departure_delay(dataframe):
    # Filter flights with no departure delay
    ontime_departures = dataframe.filter(col("DEP_DELAY") <= 0)

    # Calculate the maximum arrival delay
    max_arr_delay = (
        ontime_departures.agg(spark_max(col("ARR_DELAY")).alias("max_arr_delay"))
        .collect()[0]["max_arr_delay"]
    )
    print(f"Maximum Arrival Delay: {max_arr_delay:.2f} minutes")


    return max_arr_delay
max_delay = max_arrival_delay_no_departure_delay(df)

Maximum Arrival Delay: 701.00 minutes


Assumptions

1)The dataset columns include ARR_DELAY, DEP_DELAY, DEP_TIME, AIR_TIME, and DISTANCE.
2)DEP_TIME is in 24-hour numeric format (e.g., 900 = 9:00 AM).
3)Negative delay values indicate early departures or arrivals.
4)Flights with missing or null delay values are ignored in calculations.
5)All time values are in minutes.